In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

basePath="/Volumes/workspace/default/dataflowmed"
rawPath = "/Volumes/workspace/default/dataflowmed/raw"
bronzePath=f"{basePath}/bronze"
silverPath=f"{basePath}/silver"

In [0]:
def readDelta(tableName):
    return spark.read.format("delta").load(f"{bronzePath}/{tableName}")

def writeDelta(df,tableName):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silverPath}/{tableName}")

In [0]:
def standardizeColumns(df):
    for column in df.columns:
        df=df.withColumnRenamed(column,column.strip())
    return df

def removeDuplicates(df):
    return df.dropDuplicates()

def cleanData(df):
    df=standardizeColumns(df)
    df=removeDuplicates(df)
    return df

In [0]:
customerDf=readDelta("olist_customers_dataset")
customerDf=cleanData(customerDf)
display(customerDf)

In [0]:
def handleNulls(df,requiredColumns):
    return df.dropna(subset=requiredColumns)


def validateSchema(df):
    return df


def processSilver(df,requiredColumns):
    df=cleanData(df)
    df=handleNulls(df,requiredColumns)
    df=validateSchema(df)
    return df

In [0]:
customerDf=readDelta("olist_customers_dataset")
customerDf=processSilver(customerDf,["customer_id"])
writeDelta(customerDf,"olist_customers_dataset")

In [0]:
ordersDf=readDelta("olist_orders_dataset")
ordersDf=processSilver(ordersDf,["order_id","customer_id"])
writeDelta(ordersDf,"olist_orders_dataset")

In [0]:
productsDf=readDelta("olist_products_dataset")
productsDf=processSilver(productsDf,["product_id"])
writeDelta(productsDf,"olist_products_dataset")

In [0]:
orderItemsDf = readDelta("olist_order_items_dataset")
orderItemsDf = processSilver(orderItemsDf,["order_id", "product_id", "seller_id"])
writeDelta(orderItemsDf,"olist_order_items_dataset")

In [0]:
paymentsDf=readDelta("olist_order_payments_dataset")
paymentsDf=processSilver(paymentsDf,["order_id"])
writeDelta(paymentsDf,"olist_order_payments_dataset")

In [0]:
reviewsDf=readDelta("olist_order_reviews_dataset")
reviewsDf=processSilver(reviewsDf,["review_id","order_id"])
writeDelta(reviewsDf,"olist_order_reviews_dataset")

In [0]:
sellersDf=readDelta("olist_sellers_dataset")
sellersDf=processSilver(sellersDf,["seller_id"])
writeDelta(sellersDf,"olist_sellers_dataset")

In [0]:
geolocationDf=readDelta("olist_geolocation_dataset")
geolocationDf=processSilver(geolocationDf,["geolocation_zip_code_prefix"])
writeDelta(geolocationDf,"olist_geolocation_dataset")

In [0]:
categoryDf=readDelta("product_category_name_translation")
categoryDf=processSilver(categoryDf,["product_category_name"])
writeDelta(categoryDf,"product_category_name_translation")

In [0]:
display(dbutils.fs.ls(silverPath))

In [0]:
def validateSchema(sourceDf,targetDf=None):
    if targetDf is None:
        return sourceDf
    sourceColumns=set(sourceDf.columns)
    targetColumns=set(targetDf.columns)

    newColumns=sourceColumns-targetColumns
    removedColumns=targetColumns-sourceColumns

    if newColumns:
        print(f"New Columns : {newColumns}")

    if removedColumns:
        print(f"Removed Columns : {removedColumns}")

    return sourceDf

In [0]:
def processSilver(df,requiredColumns,targetDf=None):

    df=cleanData(df)
    df=handleNulls(df,requiredColumns)
    df=validateSchema(df,targetDf)
    return df

In [0]:
customerDf=readDelta("olist_customers_dataset")

customerDf=processSilver(customerDf,["customer_id"])

display(customerDf.limit(10))

In [0]:
def compareSchema(sourceDf,targetDf):

    sourceColumns=set(sourceDf.columns)
    targetColumns=set(targetDf.columns)

    newColumns=sourceColumns-targetColumns
    removedColumns=targetColumns-sourceColumns

    return newColumns,removedColumns

In [0]:
silverCustomerDf=readDelta("olist_customers_dataset")

newColumns,removedColumns=compareSchema(
    customerDf,
    silverCustomerDf
)

print(f"New Columns : {newColumns}")
print(f"Removed Columns : {removedColumns}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def addScdColumns(df):

    return df \
        .withColumn("effectiveDate", current_timestamp()) \
        .withColumn("expiryDate", lit(None).cast("timestamp")) \
        .withColumn("isCurrent", lit(True))

In [0]:
customerDf=readDelta("olist_customers_dataset")

customerDf=processSilver(
    customerDf,
    ["customer_id"]
)

customerDf=addScdColumns(customerDf)

writeDelta(customerDf, "olist_customers_dataset")

display(customerDf)

In [0]:
def readSilver(tableName):
    return spark.read.format("delta").load(f"{silverPath}/{tableName}")

In [0]:
def alignSchemas(df, referenceColumns):
    
    for c in referenceColumns:
        if c not in df.columns:
            df=df.withColumn(c, lit(None))
    return df.select(referenceColumns)

In [0]:
from functools import reduce

def scdType2Merge(existingDf,incomingDf,keyColumn,trackedColumns):
    newCols,removedCols=compareSchema(incomingDf,existingDf)

    if newCols:
        print(f"Schema drift detected - new columns: {newCols}")
    if removedCols:
        print(f"Schema drift detected - missing columns: {removedCols}")

    trackedColumns=list(trackedColumns)+[c for c in newCols if c not in trackedColumns]

    referenceColumns=existingDf.columns+[c for c in incomingDf.columns if c not in existingDf.columns]

    existingDf=alignSchemas(existingDf,referenceColumns)
    incomingDf=alignSchemas(incomingDf,referenceColumns)

    currentDf=existingDf.filter(col("isCurrent")==True)

    comparisonDf=incomingDf.alias("new").join(
        currentDf.alias("old"),
        on=keyColumn,
        how="left"
    )

    changeCondition=reduce(
        lambda a,b: a|b,
        [col(f"new.{c}")!=col(f"old.{c}") for c in trackedColumns]
    )

    changedDf=comparisonDf.filter(col(f"old.{keyColumn}").isNotNull()&changeCondition)
    newRowsDf=comparisonDf.filter(col(f"old.{keyColumn}").isNull())

    expiredDf=existingDf.join(
        changedDf.select(keyColumn),keyColumn,"leftsemi"
    ).withColumn("isCurrent",lit(False)) \
     .withColumn("expiryDate",current_timestamp())

    newVersionDf=changedDf.select("new.*")
    brandNewDf=newRowsDf.select("new.*")

    unchangedCurrentDf=existingDf.join(
        changedDf.select(keyColumn),keyColumn,"leftanti"
    )

    finalDf=unchangedCurrentDf \
        .unionByName(expiredDf) \
        .unionByName(newVersionDf) \
        .unionByName(brandNewDf)

    return finalDf

In [0]:
existingDf=readSilver("olist_customers_dataset")

incomingDf=spark.read \
    .option("header",True) \
    .option("inferSchema",True) \
    .csv(f"{rawPath}/olist_customers_dataset_v2")

incomingDf=processSilver(incomingDf,["customer_id"])
incomingDf=addScdColumns(incomingDf)

finalCustomerDf=scdType2Merge(
    existingDf,
    incomingDf,
    keyColumn="customer_id",
    trackedColumns=["customer_city","customer_state"]
)

display(finalCustomerDf)

In [0]:
writeDelta(finalCustomerDf, "olist_customers_dataset")

display(spark.read.format("delta").load(f"{silverPath}/olist_customers_dataset").orderBy("customer_id"))